# Day 15 — Spaced Repetition

> ⚠️ **Why this matters.** This is the trick behind Anki, Duolingo, and every effective language-learning app: **show the user the words they're about to forget, not the words they already know.** You'll implement a working scheduler today.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/15-spaced-repetition.ipynb)

## What you'll do today

**Time:** 90 min lesson + 90 min build + 30 min quiz.

- [ ] You understand the SM-2 algorithm (the simplified version Anki uses)
- [ ] You can implement a scheduler as a class
- [ ] Your english-helper now picks 'words to review today'
- [ ] You've added persistence for review state per word

## The mental model — Ebbinghaus forgetting curve

```
memory strength
|\
| \
|  \___
|      \______
+----------------> time
1d  3d   7d   14d
```

You learn a word today. Tomorrow you've forgotten ~50%. By day 7, ~80%. But: **after each successful recall, the curve gets shallower**.

**Spaced repetition** = re-test you right BEFORE you'd forget. Each successful recall pushes the next review further out.

Simplified SM-2 (what we'll implement):

| Recall result | Next review |
|---------------|-------------|
| Got it wrong | Tomorrow |
| Right | (Last interval × 2.5), max 6 months |

> 💡 **In the wild:** Anki has 100M+ active reviews per day worldwide using a variant of this exact algorithm. You're learning the foundation.

## 1. Per-word review state

Each word needs three new fields:

```python
from datetime import datetime
from dataclasses import dataclass, field

@dataclass
class ReviewState:
    interval_days: float = 1.0       # next review interval
    ease: float = 2.5                # difficulty multiplier
    last_reviewed: datetime | None = None
    next_review: datetime | None = None
    correct_count: int = 0
    wrong_count: int = 0
```

## 2. The scheduler

In [ ]:
from datetime import datetime, timedelta
from dataclasses import dataclass

@dataclass
class Scheduler:
    def record_review(self, state, correct: bool) -> None:
        state.last_reviewed = datetime.now()
        if correct:
            state.correct_count += 1
            state.interval_days = min(state.interval_days * state.ease, 180.0)
        else:
            state.wrong_count += 1
            state.interval_days = 1.0
            state.ease = max(state.ease - 0.2, 1.3)
        state.next_review = datetime.now() + timedelta(days=state.interval_days)

    def due_today(self, states) -> list:
        now = datetime.now()
        return [s for s in states if not s.next_review or s.next_review <= now]

**The schedule in action:**

- Day 0: add 'thorough'. interval=1, next_review=tomorrow.
- Day 1: review, get it right. interval = 1×2.5 = 2.5 days. next=Day 3-4.
- Day 3: right again. interval = 2.5×2.5 = 6.25 days. next≈Day 10.
- Day 10: right. interval = 15. next≈Day 25.
- Day 25: WRONG. Reset interval=1. next=tomorrow. ease drops.

**That's it.** 30 lines of code. Anki's algorithm is a refined version of this.

## End-of-day mini-project

> 🎯 **Add spaced repetition to english-helper.**

### Steps

1. Create `src/english_helper/scheduler.py` with `ReviewState` and `Scheduler` dataclasses.
2. Add `review_state` field to `Word`? **Bad idea** — Word is frozen and shouldn't grow. Instead: store `ReviewState` in a sibling dict in `WordStore`:
   ```python
   @dataclass
   class WordStore:
       words: dict[str, Word] = field(default_factory=dict)
       review: dict[str, ReviewState] = field(default_factory=dict)
   ```
3. New CLI commands:
   - `review` — quiz user on words due today
   - `due` — show count of words due today
   - `stats` — extend with avg ease, total reviews
4. Persist ReviewStates to the JSON file alongside words.

### Verify

```bash
$ english-helper add thorough
Added.
$ english-helper due
1 word due today: thorough
$ english-helper review
IPA for 'thorough': /ˈθʌrə/
✓ Correct! Next review in 2.5 days.
Done. (1/1)
```

## Connect to the project

> 🎯 **End of Phase 1 Week 3.** Your english-helper is now a real spaced-repetition tool. **Tomorrow Week 4 begins:** testing. You'll add pytest tests for everything you've built. Day 16 onward you'll feel why frozen dataclasses + ABCs + clear module boundaries pay off — they're easy to test.

**Quiz:** [15-spaced-repetition-quiz.ipynb](15-spaced-repetition-quiz.ipynb)